# Probiotic Microbiome -> Poultry Performance: ML Demo

**Purpose of this notebook**

This is a small, self-initiated exploratory project built to demonstrate applied
understanding of the intersection between probiotic genomics, poultry gut
microbiome data, and machine learning — inspired by Dr. Shakira Ghazanfar's
published work on multi-strain probiotics and poultry performance
(e.g., her *Pediococcus acidilactici* broiler study) and her `ProbioPred`
ML tool for probiotic prediction.

**Important framing note:** this notebook uses a **synthetic (simulated)
dataset** with a known, deliberately built-in ground truth. This is
intentional — it lets us verify that every step of the pipeline (filtering,
normalization, CLR transformation, model training, feature importance) is
implemented *correctly*, before trusting it on real biological data where
there's no answer key to check against.

**What this notebook covers, end to end:**
1. Simulating realistic microbiome count data (with the quirks real data has:
   variable sequencing depth, sparsity/zeros, compositionality)
2. Preprocessing: filtering rare taxa, normalizing, CLR transformation
3. Training a Random Forest to predict poultry performance (FCR, weight gain)
   from microbiome composition
4. Validating with 5-fold cross-validation (appropriate for small sample sizes)
5. Extracting feature importance to identify which taxa the model finds
   most predictive
6. Checking the model's findings against the known ground truth


## 1. Generating the Synthetic Dataset

### Why synthetic data, and why these specific design choices

Real poultry microbiome studies are typically small (tens of birds, not
thousands) because animal feeding trials are expensive and slow. We simulate
**60 birds** to reflect that realistic scale.

We simulate **20 bacterial taxa** (columns) per bird. This mimics a real gut
microbiome table's structure: rows = samples (birds), columns = taxa,
values = sequencing read counts.

**The key idea — we bake in a known ground truth:**

| Taxon | Role | Effect |
|---|---|---|
| `Taxon_03` | Beneficial (strong) | Higher abundance -> **lower FCR**, **higher weight gain** |
| `Taxon_07` | Harmful | Higher abundance -> **higher FCR**, **lower weight gain** |
| `Taxon_12` | Beneficial (weak) | Same direction as Taxon_03, but a smaller effect |
| All other 17 taxa | Noise | No real relationship to outcomes — pure randomness |

This lets us later check: *does the Random Forest's feature importance
correctly identify Taxon_03, Taxon_07, and Taxon_12 as the most important
taxa, out of 20?* If yes, that's strong evidence the whole pipeline
(preprocessing + modeling + interpretation) is working correctly.

### Realistic quirks we deliberately include

- **Variable sequencing depth per bird** — in real sequencing, different
  samples get different total read counts for purely technical reasons,
  unrelated to biology. This is *why* normalization is a necessary step,
  not an arbitrary one.
- **Sparsity (many zeros)** — real microbiome data has many taxa absent
  in many individual samples. This is *why* we need a pseudocount before
  taking logs later (log(0) is undefined).
- **Irreducible biological noise** — even the true drivers (Taxon_03,
  Taxon_07, Taxon_12) don't perfectly determine the outcome; we add random
  noise on top, since real biology is never perfectly deterministic. This
  keeps the demo honest rather than artificially "too clean."


In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)  # reproducibility -- same random numbers every run

N_BIRDS = 60
N_TAXA = 20

taxon_names = [f"Taxon_{i:02d}" for i in range(1, N_TAXA + 1)]
print(f"Simulating {N_BIRDS} birds x {N_TAXA} taxa")

Simulating 60 birds x 20 taxa


In [2]:
# ---- Step A: simulate raw read counts per bird per taxon ----

# Each taxon has its own "typical" abundance level (some bacteria are
# naturally more common in a chicken gut than others).
base_abundance = np.random.gamma(shape=2.0, scale=1.5, size=N_TAXA)

raw_counts = np.zeros((N_BIRDS, N_TAXA))
for j in range(N_TAXA):
    # per-bird random variation around that taxon's typical level
    lam = base_abundance[j] * np.random.uniform(0.5, 2.0, size=N_BIRDS)
    raw_counts[:, j] = np.random.poisson(lam * 50)

# Force realistic sparsity: randomly zero-out ~30% of entries.
# (In real data, most bacteria are simply absent in most individual samples.)
zero_mask = np.random.rand(N_BIRDS, N_TAXA) < 0.30
raw_counts[zero_mask] = 0

# Give each bird a different total sequencing depth -- purely technical
# variation, nothing to do with the bird's actual biology.
depth_multiplier = np.random.uniform(0.5, 2.0, size=N_BIRDS)
raw_counts = (raw_counts * depth_multiplier[:, None]).astype(int)

df_counts = pd.DataFrame(raw_counts, columns=taxon_names)
df_counts.insert(0, "Bird_ID", [f"Bird_{i+1:02d}" for i in range(N_BIRDS)])

print("Total read depth per bird (first 10 birds) -- notice these differ:")
print(df_counts[taxon_names].sum(axis=1).head(10).values)

Total read depth per bird (first 10 birds) -- notice these differ:
[1361 1671 4071 2769 2124 2139 2098 5225 2678 6252]


### Simulating the outcomes (FCR and Weight Gain)

Now we generate the two performance metrics using the ground-truth
relationship described above. Note we use **true relative abundance**
(each taxon's share of that bird's total reads) to drive the biology —
this reflects reality: an animal's physiology responds to the actual
proportion of bacteria present, not to an arbitrary raw sequencing count
that depends on how deeply that sample happened to be sequenced.


In [3]:
# ---- Step B: true relative abundance (drives the actual biology) ----
rel_abundance = raw_counts / raw_counts.sum(axis=1, keepdims=True)

good_a = rel_abundance[:, 2]   # Taxon_03 (index 2)  -> beneficial, strong
bad_bug = rel_abundance[:, 6]  # Taxon_07 (index 6)  -> harmful
good_b = rel_abundance[:, 11]  # Taxon_12 (index 11) -> beneficial, weak

# ---- Step C: simulate outcomes ----

# FCR (Feed Conversion Ratio): lower = better (less feed per kg of gain).
# Baseline ~2.0, pulled down by good bugs, pushed up by the bad bug.
fcr_noise = np.random.normal(0, 0.08, size=N_BIRDS)  # irreducible biological noise
fcr = 2.0 - (good_a * 3.5) - (good_b * 1.5) + (bad_bug * 4.0) + fcr_noise
fcr = np.clip(fcr, 1.2, 3.0)  # realistic FCR range for broilers

# Weight gain (grams): baseline ~1500g, pushed up by good bugs, down by bad bug.
wg_noise = np.random.normal(0, 60, size=N_BIRDS)
weight_gain = 1500 + (good_a * 2500) + (good_b * 1000) - (bad_bug * 3000) + wg_noise
weight_gain = np.clip(weight_gain, 900, 2200)

df_outcomes = pd.DataFrame({
    "Bird_ID": df_counts["Bird_ID"],
    "FCR": fcr.round(3),
    "Weight_Gain_g": weight_gain.round(1),
})

print(df_outcomes.head(10))

   Bird_ID    FCR  Weight_Gain_g
0  Bird_01  1.570         1673.4
1  Bird_02  1.687         1628.1
2  Bird_03  2.090         1407.2
3  Bird_04  1.902         1470.6
4  Bird_05  2.099         1401.2
5  Bird_06  1.903         1532.6
6  Bird_07  1.672         1636.4
7  Bird_08  2.177         1237.1
8  Bird_09  2.000         1486.0
9  Bird_10  1.672         1633.4


In [4]:
# Quick look at the raw data we'll be working with
print("Raw counts shape:", df_counts.shape)
df_counts.head()

Raw counts shape: (60, 21)


,Bird_ID,Taxon_01,Taxon_02,Taxon_03,Taxon_04,Taxon_05,Taxon_06,Taxon_07,Taxon_08,Taxon_09,...,Taxon_11,Taxon_12,Taxon_13,Taxon_14,Taxon_15,Taxon_16,Taxon_17,Taxon_18,Taxon_19,Taxon_20
0,Bird_01,124,117,97,88,183,167,0,126,0,...,0,122,150,0,0,0,29,104,0,41
1,Bird_02,249,61,63,73,260,0,0,0,0,...,78,179,0,0,66,231,0,176,177,48
2,Bird_03,270,0,0,158,704,183,228,0,509,...,126,330,756,150,63,417,177,0,0,0
3,Bird_04,0,262,202,126,0,344,85,330,348,...,0,0,307,88,57,0,165,160,268,0
4,Bird_05,0,187,49,0,0,162,135,307,151,...,0,201,553,128,147,0,0,87,0,0


## 2. Preprocessing: Filtering, Normalization, CLR Transformation

Raw microbiome count data can't be fed directly into a model. We apply four
steps, in this order:

### 2.1 Filter rare/low-abundance taxa

Some taxa appear in only 1-2 samples with just a handful of reads — this is
almost certainly sequencing noise rather than real biological signal.
Keeping them just adds useless dimensions to our already-small dataset
(worsening the "more features than samples" problem). We keep a taxon only
if it's present in **at least 3 samples** and reaches **at least 0.5%**
relative abundance in at least one sample.

### 2.2 Convert to relative abundance

This corrects for the variable sequencing depth we deliberately built into
the simulation — dividing each bird's counts by that bird's total read
count puts every bird on the same scale (proportions, not raw counts that
depend on how deeply that sample happened to be sequenced).

### 2.3 Add a pseudocount

Real microbiome data has many zeros (a taxon simply absent in a sample).
We can't take `log(0)` — it's undefined. So we add a tiny constant
(`0.0001`) to every value first, purely to make the next step mathematically
possible.

### 2.4 CLR (Centered Log-Ratio) transformation

This is the key step that fixes **compositionality** — the fact that raw
percentages are all "locked together" (they must sum to 100%, so one taxon
increasing mathematically forces others to decrease, even with no real
biological connection between them).

For each bird (row), we compute:

```
CLR(taxon_i) = log( abundance(taxon_i) / geometric_mean(all taxa in this bird) )
```

Intuitively: instead of saying "Taxon_03 is 25% of this bird's gut" (a
number that's meaningless without knowing what makes up the other 75%),
we say "Taxon_03 is *this much above or below* what's typical for this
particular bird." Values above 0 mean "more abundant than this bird's
average taxon"; values below 0 mean "less abundant than average."

**Sanity check:** after CLR transformation, every row (bird) should sum to
very close to 0 — this is a direct mathematical consequence of comparing
everything to that row's own geometric mean. We verify this below.


In [5]:
taxon_cols = [c for c in df_counts.columns if c.startswith("Taxon")]
counts = df_counts[taxon_cols].values

print(f"Before filtering: {len(taxon_cols)} taxa")

Before filtering: 20 taxa


In [6]:
# ---- 2.1 Filter rare taxa ----
rel_for_filtering = counts / counts.sum(axis=1, keepdims=True)
present_in_n_samples = (counts > 0).sum(axis=0)
max_rel_abundance = rel_for_filtering.max(axis=0)

keep_mask = (present_in_n_samples >= 3) & (max_rel_abundance >= 0.005)
kept_taxa = [t for t, keep in zip(taxon_cols, keep_mask) if keep]
dropped_taxa = [t for t, keep in zip(taxon_cols, keep_mask) if not keep]

print(f"After filtering: {len(kept_taxa)} taxa kept, {len(dropped_taxa)} dropped")
if dropped_taxa:
    print(f"Dropped taxa: {dropped_taxa}")

counts_filtered = counts[:, keep_mask]

After filtering: 20 taxa kept, 0 dropped


In [7]:
# ---- 2.2 Relative abundance ----
rel_abundance_filtered = counts_filtered / counts_filtered.sum(axis=1, keepdims=True)

# ---- 2.3 Pseudocount ----
PSEUDOCOUNT = 1e-4
rel_abundance_pc = rel_abundance_filtered + PSEUDOCOUNT

# ---- 2.4 CLR transform ----
geometric_means = np.exp(np.mean(np.log(rel_abundance_pc), axis=1, keepdims=True))
clr_values = np.log(rel_abundance_pc / geometric_means)

df_clr = pd.DataFrame(clr_values, columns=kept_taxa)
df_clr.insert(0, "Bird_ID", df_counts["Bird_ID"])

print("CLR-transformed values (first 5 birds, first 6 taxa):")
df_clr.iloc[:5, :7].round(3)

CLR-transformed values (first 5 birds, first 6 taxa):


,Bird_ID,Taxon_01,Taxon_02,Taxon_03,Taxon_04,Taxon_05,Taxon_06
0,Bird_01,2.623,2.565,2.378,2.280,3.012,2.920
1,Bird_02,3.177,1.772,1.804,1.951,3.220,-4.131
2,Bird_03,2.327,-4.171,-4.171,1.793,3.285,1.939
3,Bird_04,-4.449,2.405,2.145,1.674,-4.449,2.677
4,Bird_05,-3.862,2.919,1.583,-3.862,-3.862,2.776


In [8]:
# Sanity check: each row should sum to ~0 (mathematical property of CLR)
row_sums = df_clr[kept_taxa].sum(axis=1)
print("Row sums after CLR (should all be ~0):")
print(row_sums.head(10).round(6).values)
assert np.allclose(row_sums, 0, atol=1e-6), "CLR row sums should be ~0!"
print("\nSanity check passed.")

Row sums after CLR (should all be ~0):
[-0.  0. -0. -0.  0.  0.  0.  0.  0. -0.]

Sanity check passed.


## 3. Training the Random Forest Model

### Why Random Forest for this kind of data

Random Forest is well suited to microbiome-to-phenotype prediction because:

- It handles **high-dimensional, non-linear** relationships without needing
  us to specify the functional form in advance.
- Building many trees on **random subsets of birds** (bootstrap sampling)
  and **random subsets of taxa** at each split makes it naturally resistant
  to overfitting on small datasets like ours (60 birds) — each tree only
  sees part of the picture, and averaging their predictions cancels out
  each tree's individual mistakes/noise-chasing.
- It gives us **feature importance** almost for free — a ranked list of
  which taxa the model relied on most, which is the actual scientific
  payoff (not just prediction accuracy).

We deliberately chose Random Forest over Gradient Boosting here: Gradient
Boosting builds trees sequentially, each one correcting the previous
trees' errors — powerful with large datasets, but prone to chasing
random noise as if it were real signal when the sample size is this small
(60 birds). Random Forest's independent, averaged trees are more
conservative and robust in this low-sample regime.

### Why 5-fold cross-validation

With only 60 birds, a single train/test split would leave far too few
birds in the test set to reliably judge performance (one lucky or unlucky
split could make the model look much better or worse than it really is).
5-fold cross-validation rotates through 5 train/test splits so that every
bird is used for testing exactly once, giving a much more stable estimate.


In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

df_outcomes_loaded = df_outcomes  # already in memory from Section 1
df = df_clr.merge(df_outcomes_loaded, on="Bird_ID")

X = df[kept_taxa].values
print("Feature matrix shape (birds x taxa):", X.shape)

Feature matrix shape (birds x taxa): (60, 20)


In [10]:
results = {}

for target_name in ["FCR", "Weight_Gain_g"]:
    y = df[target_name].values

    model = RandomForestRegressor(
        n_estimators=200,
        max_features="sqrt",   # each split only considers a random subset of taxa
        random_state=42,
    )

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    neg_mae_scores = cross_val_score(model, X, y, cv=kf, scoring="neg_mean_absolute_error")
    mae_scores = -neg_mae_scores
    r2_scores = cross_val_score(model, X, y, cv=kf, scoring="r2")

    print(f"=== Target: {target_name} ===")
    print(f"Cross-validated MAE per fold: {mae_scores.round(3)}")
    print(f"Mean MAE: {mae_scores.mean():.3f} (+/- {mae_scores.std():.3f})")
    print(f"Mean R^2:  {r2_scores.mean():.3f} (+/- {r2_scores.std():.3f})")
    print()

    # Fit on all data to extract feature importance (CV was only for
    # honestly estimating performance; the final importance analysis
    # uses every bird we have)
    model.fit(X, y)
    importances = pd.Series(model.feature_importances_, index=kept_taxa).sort_values(ascending=False)
    results[target_name] = importances

    print(f"Top 5 most important taxa for {target_name}:")
    print(importances.head(5).round(4))
    print("\n" + "-"*50 + "\n")

=== Target: FCR ===
Cross-validated MAE per fold: [0.08  0.126 0.133 0.104 0.1  ]
Mean MAE: 0.109 (+/- 0.019)
Mean R^2:  0.125 (+/- 0.120)

Top 5 most important taxa for FCR:
Taxon_03    0.1336
Taxon_07    0.1192
Taxon_12    0.0854
Taxon_08    0.0721
Taxon_14    0.0528
dtype: float64

--------------------------------------------------



=== Target: Weight_Gain_g ===
Cross-validated MAE per fold: [92.496 79.431 83.477 68.307 90.532]
Mean MAE: 82.849 (+/- 8.668)
Mean R^2:  0.084 (+/- 0.210)



Top 5 most important taxa for Weight_Gain_g:
Taxon_07    0.1360
Taxon_03    0.1215
Taxon_09    0.0811
Taxon_19    0.0640
Taxon_06    0.0597
dtype: float64

--------------------------------------------------



### Interpreting the R² and MAE values

Don't be alarmed that R² is modest (roughly 0.08-0.15) rather than close to
1.0 — this is expected and intentional. We added realistic irreducible
biological noise on top of the true signal (real animals never respond
perfectly uniformly to the same microbiome composition). A very high R² on
a toy dataset like this would actually be a red flag suggesting the
simulation is unrealistically clean, not a sign of a better model.

The more meaningful check is whether the model's **feature importance**
correctly recovers the taxa we know are the true drivers — that's what we
check next.


## 4. Validating Against Ground Truth

Since we built this dataset ourselves, we know the correct answer:
`Taxon_03` (strong beneficial), `Taxon_07` (harmful), and `Taxon_12` (weak
beneficial) should rank as the most important features, while the
remaining 17 taxa are pure noise and should rank low.

This step is the actual point of using synthetic data first: if the
pipeline can correctly recover a known answer, that's strong evidence the
preprocessing and modeling steps are implemented correctly — giving
confidence before applying the same pipeline to real data, where there's
no answer key to check against.


In [11]:
expected_drivers = {"Taxon_03", "Taxon_07", "Taxon_12"}

for target_name, importances in results.items():
    top5 = set(importances.head(5).index)
    found = top5 & expected_drivers
    print(f"{target_name}:")
    print(f"  Top 5 ranked taxa: {sorted(top5)}")
    print(f"  Ground-truth drivers found in top 5: {sorted(found)} ({len(found)}/3)")
    print()

FCR:
  Top 5 ranked taxa: ['Taxon_03', 'Taxon_07', 'Taxon_08', 'Taxon_12', 'Taxon_14']
  Ground-truth drivers found in top 5: ['Taxon_03', 'Taxon_07', 'Taxon_12'] (3/3)

Weight_Gain_g:
  Top 5 ranked taxa: ['Taxon_03', 'Taxon_06', 'Taxon_07', 'Taxon_09', 'Taxon_19']
  Ground-truth drivers found in top 5: ['Taxon_03', 'Taxon_07'] (2/3)



### What this result tells us

If the true drivers rank at or near the top (as they should for FCR, and
largely for Weight Gain), this confirms the full pipeline — filtering,
relative abundance normalization, CLR transformation, Random Forest
training, and feature importance extraction — is working as intended.

Note that weaker effects (like `Taxon_12`'s smaller beneficial effect) may
not always surface reliably in the top rankings for every outcome — this is
a realistic and honest limitation, not a flaw: detecting weaker signals
reliably typically requires larger sample sizes than a 60-bird trial can
provide. This is a genuine statistical constraint worth being upfront about
when discussing the pipeline's capabilities.


## 5. Next Steps

This notebook validates the pipeline on synthetic data with a known answer.
The logical next steps, in order of increasing scope:

1. **Apply this same, now-validated pipeline to a real public dataset** —
   ideally one that already provides a processed abundance table (not raw
   sequencing reads), paired with real performance metrics.
2. **Process raw sequencing reads from scratch** (quality control, ASV
   inference via DADA2, taxonomic classification against a reference
   database like SILVA) to generate an abundance table directly from public
   raw data on NCBI SRA — a substantially larger undertaking, typically
   handled with dedicated tools (QIIME2, DADA2) rather than from-scratch
   Python code.
3. **Extend interpretability with SHAP values** for per-bird, per-taxon
   explanations, rather than only a single global importance ranking.

This notebook intentionally stops at step 1's foundation — the emphasis was
on building and *verifying* a correct pipeline end-to-end, rather than
rushing into real data without first confirming the method works.
